In [186]:
import dgl
from pathlib import Path



In [31]:
graphs_path = "../../fraud_detection/data/graphs_train_val_test.bin"
[G, G2, G3], _ = dgl.load_graphs(graphs_path)

In [131]:
train_mask = G.ndata["mask"]
val_mask = G2.ndata["mask"]
test_mask = G3.ndata["mask"]

In [4]:
G

Graph(num_nodes=1479770, num_edges=37052682,
      ndata_schemes={'mask': Scheme(shape=(), dtype=torch.uint8), 'userid': Scheme(shape=(), dtype=torch.int64), 'labels': Scheme(shape=(1,), dtype=torch.float32), 'features': Scheme(shape=(224,), dtype=torch.float32)}
      edata_schemes={})

In [5]:
import os
import torch
import numpy as np


import dgl.graphbolt as gb

In [21]:
dataset = gb.BuiltinDataset("ogbn-arxiv-seeds").load()

datasets/ogbn-arxiv-seeds.zip: 100%|██████████| 177M/177M [00:02<00:00, 76.2MB/s] 


Extracting file to datasets
The dataset is already preprocessed.


In [6]:
dataset.tasks

[OnDiskTask(validation_set=ItemSet(
                items=(tensor([    57698,    145220,    599977,  ..., 110949043, 110952576,
                    110996715], dtype=torch.int32), tensor([164,  64, 168,  ...,   5, 134,  18])),
                names=('seeds', 'labels'),
            ),
            train_set=ItemSet(
                items=(tensor([      602,       684,      1384,  ..., 111048965, 111049596,
                    111052523], dtype=torch.int32), tensor([ 79,  26,  80,  ..., 121,  79,  76])),
                names=('seeds', 'labels'),
            ),
            test_set=ItemSet(
                items=(tensor([    43768,    122141,    251239,  ..., 111059783, 111059927,
                    111059953], dtype=torch.int32), tensor([ 82, 101,  60,  ...,  19,  55, 157])),
                names=('seeds', 'labels'),
            ),
            metadata={'name': 'node_classification', 'num_classes': 172},)]

In [7]:
dataset.tasks[0]

OnDiskTask(validation_set=ItemSet(
               items=(tensor([    57698,    145220,    599977,  ..., 110949043, 110952576,
                   110996715], dtype=torch.int32), tensor([164,  64, 168,  ...,   5, 134,  18])),
               names=('seeds', 'labels'),
           ),
           train_set=ItemSet(
               items=(tensor([      602,       684,      1384,  ..., 111048965, 111049596,
                   111052523], dtype=torch.int32), tensor([ 79,  26,  80,  ..., 121,  79,  76])),
               names=('seeds', 'labels'),
           ),
           test_set=ItemSet(
               items=(tensor([    43768,    122141,    251239,  ..., 111059783, 111059927,
                   111059953], dtype=torch.int32), tensor([ 82, 101,  60,  ...,  19,  55, 157])),
               names=('seeds', 'labels'),
           ),
           metadata={'name': 'node_classification', 'num_classes': 172},)

In [8]:
dataset.feature

TorchBasedFeatureStore(
    {(<OnDiskFeatureDataDomain.NODE: 'node'>, None, 'year'): TorchBasedFeature(
        feature=tensor([[2012],
                        [2013],
                        [1988],
                        ...,
                        [2020],
                        [1997],
                        [2020]]),
        metadata={},
    ), (<OnDiskFeatureDataDomain.NODE: 'node'>, None, 'label'): TorchBasedFeature(
        feature=tensor([[ nan],
                        [ nan],
                        [ nan],
                        ...,
                        [157.],
                        [ nan],
                        [ nan]]),
        metadata={},
    ), (<OnDiskFeatureDataDomain.NODE: 'node'>, None, 'feat'): TorchBasedFeature(
        feature=tensor([[-0.4340, -0.3536, -0.3905,  ...,  0.1789, -0.2058, -0.0182],
                        [-0.2557,  0.0541,  0.1021,  ..., -0.1388, -0.2668, -0.2331],
                        [-0.0451,  0.2022, -0.0104,  ...,  0.0784, -0.0

In [9]:
graph = dataset.graph
feature = dataset.feature
train_set = dataset.tasks[0].train_set
valid_set = dataset.tasks[0].validation_set
test_set = dataset.tasks[0].test_set
task_name = dataset.tasks[0].metadata["name"]
num_classes = dataset.tasks[0].metadata["num_classes"]

print(f"Task: {task_name}. Number of classes: {num_classes}")

Task: node_classification. Number of classes: 172


In [10]:
dataset.tasks[0].metadata

{'name': 'node_classification', 'num_classes': 172}

Let’s say that each node will gather messages from 4 neighbors on each layer. The code defining the data loader and neighbor sampler will look like the following.



In [11]:
device = torch.device("cuda", 7)
datapipe = gb.ItemSampler(train_set, batch_size=1024, shuffle=True)
datapipe = datapipe.sample_neighbor(graph, [4, 4])
datapipe = datapipe.fetch_feature(feature, node_feature_keys=["feat"])
datapipe = datapipe.copy_to(device)
train_dataloader = gb.DataLoader(datapipe, num_workers=0)

You can iterate over the data loader and a `MiniBatch` object is yielded.



In [12]:
data = next(iter(train_dataloader))

print(data)

MiniBatch(seeds=tensor([11690124, 51052647, 57528820,  ..., 48947313, 14600020, 40797893],
                       device='cuda:7', dtype=torch.int32),
          sampled_subgraphs=[SampledSubgraphImpl(sampled_csc=CSCFormatBase(indptr=tensor([    0,     1,     5,  ..., 19207, 19211, 19215], device='cuda:7',
                                                                                       dtype=torch.int32),
                                                                         indices=tensor([ 1024,  4872,  4873,  ..., 22124, 22125, 22126], device='cuda:7',
                                                                                        dtype=torch.int32),
                                                           ),
                                               original_row_node_ids=tensor([11690124, 51052647, 57528820,  ..., 54916816, 11584565, 85590085],
                                                                            device='cuda:7', dtype=torch.int32),
    

You can get the input node IDs from MFGs.



In [13]:
message_flow_graphs = data.blocks
input_nodes = message_flow_graphs[0].srcdata[dgl.NID]
print(f"Input nodes: {input_nodes}.")


Input nodes: tensor([11690124, 51052647, 57528820,  ..., 54916816, 11584565, 85590085],
       device='cuda:7', dtype=torch.int32).


# Defining Model

In [15]:
import torch.nn as nn
import torch.nn.functional as F
# os.environ['DGLBACKEND'] = "pytorch"

import dgl.nn.pytorch as dglnn

class Model(nn.Module):
    def __init__(self, in_features, hidden_features, num_classes) -> None:
        super().__init__()
        
        self.conv1 = dglnn.SAGEConv(in_features, hidden_features, aggregator_type="mean")
        self.conv2 = dglnn.SAGEConv(hidden_features, num_classes, aggregator_type="mean")
        
        self.hidden_features = hidden_features
    
    def forward(self, message_flow_graphs, x):
        
        h = self.conv1(message_flow_graphs[0], x)
        h = F.relu(h)
        h = self.conv2(message_flow_graphs[1], h)
        return h

in_size = feature.size("node", None, "feat")[0]
model = Model(in_features=in_size, hidden_features=64, num_classes=num_classes).to(device)

# Training loop

In [16]:
from sklearn.metrics import accuracy_score


from torchmetrics import Accuracy
from tqdm import tqdm

In [17]:
opt = torch.optim.Adam(model.parameters())

datapipe = gb.ItemSampler(valid_set, batch_size=1024, shuffle=False)
datapipe = datapipe.sample_neighbor(graph, [4, 4])
datapipe = datapipe.fetch_feature(feature, node_feature_keys=["feat"])
datapipe = datapipe.copy_to(device)
valid_dataloader = gb.DataLoader(datapipe, num_workers=0)

In [18]:
EPOCHS = 10

metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)

for epoch in range(EPOCHS):
    metric = metric.to(device)

    model.train()
    
    with tqdm(train_dataloader) as tq:
        
        for step, data in enumerate(tq):
            
            x = data.node_features["feat"]
            labels = data.labels
            
            predictions = model(data.blocks, x)
            
            loss = F.cross_entropy(predictions, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            accuracy = metric(predictions, labels)
            
            tq.set_postfix(
                {"loss": f"{loss.detach()}", "acc": f"{accuracy.detach()}"}
            )
            
    model.eval()
    predictions = []
    labels = []
    
    metric = metric.to("cpu")
    with tqdm(valid_dataloader) as tq, torch.no_grad(): # two context managers
        for data in tq:
            x = data.node_features["feat"]
            labels.append(data.labels.detach().cpu())
            predictions.append(model(data.blocks, x).argmax(1).detach().cpu())
        predictions = torch.concatenate(predictions)
        labels = torch.concatenate(labels)
        accuracy = metric(predictions, labels)
        print("Epoch {} Validation Accuracy {}".format(epoch, accuracy))

        # Note that this tutorial do not train the whole model to the end.

1179it [07:31,  2.61it/s, loss=1.4388189315795898, acc=0.5556780695915222]
123it [00:46,  2.64it/s]


Epoch 0 Validation Accuracy 0.5473915338516235


1179it [01:03, 18.56it/s, loss=1.3220889568328857, acc=0.5953693389892578]
123it [00:06, 18.91it/s]


Epoch 1 Validation Accuracy 0.5719554424285889


1179it [01:03, 18.48it/s, loss=1.313643455505371, acc=0.6052922010421753]
123it [00:06, 19.23it/s]


...

Epoch 7 Validation Accuracy 0.6075440049171448


1179it [01:01, 19.02it/s, loss=1.1930367946624756, acc=0.6207276582717896]
123it [00:06, 19.65it/s]


Epoch 8 Validation Accuracy 0.5961122512817383


1179it [01:02, 18.74it/s, loss=1.2198256254196167, acc=0.6251378059387207]
123it [00:06, 18.44it/s]


Epoch 9 Validation Accuracy 0.60029536485672


## Cooking graphbolt for my code

In [6]:
G

Graph(num_nodes=1479770, num_edges=37052682,
      ndata_schemes={'mask': Scheme(shape=(), dtype=torch.uint8), 'userid': Scheme(shape=(), dtype=torch.int64), 'labels': Scheme(shape=(1,), dtype=torch.float32), 'features': Scheme(shape=(224,), dtype=torch.float32)}
      edata_schemes={})

## Homogeneous graph

In [148]:
base_dir = "./afraud_homogeneius_graph"
os.makedirs(base_dir, exist_ok=True)
print(f"Created base directory: {base_dir}")
import pandas as pd

Created base directory: ./afraud_homogeneius_graph


### Graph structure

In [149]:
num_nodes = G.num_nodes()
num_edges = G.num_edges()

edges_path = os.path.join(base_dir, "edges.csv")
edges = torch.stack(G.edges()).T
print(f"Part of edges:\n{edges[:5, :]}")


df = pd.DataFrame(edges)
df.to_csv(edges_path, index=False, header=False)
print(f"Edges are saved into {edges_path}")

Part of edges:
tensor([[      0, 1463416],
        [      0, 1447967],
        [      0, 1419323],
        [      0, 1337473],
        [      0, 1331591]], dtype=torch.int32)
Edges are saved into ./afraud_homogeneius_graph/edges.csv


### Features


In [150]:
node_features = G.ndata["features"].numpy()
features_path = os.path.join(base_dir, "node_features.npy")
print(f"Features were stored in {features_path}")
np.save(features_path, node_features)

Features were stored in ./afraud_homogeneius_graph/node_features.npy


## Node classification task

In [151]:
# split indices
indices = np.arange(num_nodes)
print(f"{len(indices)}")

1479770


In [152]:
train_mask = train_mask.bool()
val_mask = val_mask.bool()
test_mask = test_mask.bool()

In [153]:
train_ids_path = os.path.join(base_dir, "train_indices.npy")
val_ids_path = os.path.join(base_dir, "val_indices.npy")
test_ids_path = os.path.join(base_dir, "test_indices.npy")


train_offset = int(0.8 * len(indices))
val_offset = int(train_offset + (0.1 * len(indices)))


train_indices = indices[:train_offset]
val_indices = indices[train_offset:val_offset]
test_indices = indices[val_offset:]


# train_indices = indices[train_mask].astype(np.int32)
# val_indices = indices[val_mask].astype(np.int32)
# test_indices = indices[test_mask].astype(np.int32)


np.save(train_ids_path, train_indices)
np.save(val_ids_path, val_indices)
np.save(test_ids_path, test_indices)


In [170]:
labels = G.ndata["labels"].numpy().reshape(-1)



train_labels_path = os.path.join(base_dir, "train_labels.npy")
val_labels_path = os.path.join(base_dir, "val_labels.npy")
test_labels_path = os.path.join(base_dir, "test_labels.npy")

train_labels = labels[train_indices].astype(np.int64)
val_labels = labels[val_indices].astype(np.int64)
test_labels = labels[test_indices].astype(np.int64)


np.save(train_labels_path, train_labels)
np.save(val_labels_path, val_labels)
np.save(test_labels_path, test_labels)


### Organize Data into YAML file

In [171]:
yaml_content = f"""
    dataset_name: antifraud_graph
    graph:
      nodes:
        - num: {num_nodes}
      edges:
        - format: csv
          path: {os.path.basename(edges_path)}
    feature_data:
      - domain: node
        name: features
        format: numpy
        path: {os.path.basename(features_path)}
    tasks:
      - name: node_classification
        num_classes: 2
        train_set:
          - data:
              - name: seed_nodes
                format: numpy
                path: {os.path.basename(train_ids_path)}
              - name: labels
                format: numpy
                path: {os.path.basename(train_labels_path)}
        validation_set:
          - data:
              - name: seed_nodes
                format: numpy
                path: {os.path.basename(val_ids_path)}
              - name: labels
                format: numpy
                path: {os.path.basename(val_labels_path)}
        test_set:
          - data:
              - name: seed_nodes
                format: numpy
                path: {os.path.basename(test_ids_path)}
              - name: labels
                format: numpy
                path: {os.path.basename(test_labels_path)}
"""


metadata_path = os.path.join(base_dir, "metadata.yaml")
with open(metadata_path, "w") as f:
  f.write(yaml_content)

## Instantiate OnDiskDataset

In [172]:
dataset = gb.OnDiskDataset(base_dir, auto_cast_to_optimal_dtype=False).load()
graph = dataset.graph
print(f"Loaded graph: {graph}\n")

feature = dataset.feature
print(f"Loaded feature store: {feature}\n")

tasks = dataset.tasks
nc_task = tasks[0]
print(f"Loaded node classification task: {nc_task}\n")

The on-disk dataset is re-preprocessing, so the existing preprocessed dataset has been removed.
Start to preprocess the on-disk dataset.
Finish preprocessing the on-disk dataset.
Loaded graph: FusedCSCSamplingGraph(csc_indptr=tensor([       0,       59,       63,  ..., 37052680, 37052681, 37052682]),
                      indices=tensor([      0,   21004,   21439,  ..., 1479767, 1479768, 1479769]),
                      total_num_nodes=1479770, num_edges=37052682,)

Loaded feature store: TorchBasedFeatureStore(
    {(<OnDiskFeatureDataDomain.NODE: 'node'>, None, 'features'): TorchBasedFeature(
        feature=tensor([[-0.3572, -0.0143, -0.8640,  ...,  0.0000, -0.8912, -1.1457],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000, -0.8912,  0.5234],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762, -0.8675],
                        ...,
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762, -1.1457],
                        [-

## Simple training loop

In [173]:
feature

TorchBasedFeatureStore(
    {(<OnDiskFeatureDataDomain.NODE: 'node'>, None, 'features'): TorchBasedFeature(
        feature=tensor([[-0.3572, -0.0143, -0.8640,  ...,  0.0000, -0.8912, -1.1457],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000, -0.8912,  0.5234],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762, -0.8675],
                        ...,
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762, -1.1457],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762,  0.5234],
                        [-0.3572, -0.0143,  0.6343,  ...,  0.0000,  1.1762,  0.5234]]),
        metadata={},
    )}
)

In [174]:
import torch.nn as nn
import torch.nn.functional as F
# os.environ['DGLBACKEND'] = "pytorch"
import dgl.nn.pytorch as dglnn
device = torch.device("cuda", 7)

class Model(nn.Module):
    def __init__(self, in_features, hidden_features, num_classes) -> None:
        super().__init__()
        
        self.conv1 = dglnn.SAGEConv(in_features, hidden_features, aggregator_type="mean")
        self.conv2 = dglnn.SAGEConv(hidden_features, num_classes, aggregator_type="mean")
        
        self.hidden_features = hidden_features
    
    def forward(self, message_flow_graphs, x):
        
        h = self.conv1(message_flow_graphs[0], x)
        h = F.relu(h)
        h = self.conv2(message_flow_graphs[1], h)
        return h

in_size = feature.size("node", None, "features")[0]
model = Model(in_features=in_size, hidden_features=64, num_classes=2).to(device)

In [175]:
graph = dataset.graph
feature = dataset.feature
train_set = dataset.tasks[0].train_set
valid_set = dataset.tasks[0].validation_set
test_set = dataset.tasks[0].test_set
task_name = dataset.tasks[0].metadata["name"]
num_classes = dataset.tasks[0].metadata["num_classes"]

print(f"Task: {task_name}. Number of classes: {num_classes}")



Task: node_classification. Number of classes: 2


In [180]:
datapipe = gb.ItemSampler(train_set, batch_size=1000000, shuffle=True)
datapipe = datapipe.sample_neighbor(graph, [4, 4])
datapipe = datapipe.fetch_feature(feature, node_feature_keys=["features"])
datapipe = datapipe.copy_to(device)
train_dataloader = gb.DataLoader(datapipe, num_workers=0)

# Training loop

In [181]:
from sklearn.metrics import accuracy_score


from torchmetrics import Accuracy
from tqdm import tqdm

In [182]:
opt = torch.optim.Adam(model.parameters())

datapipe = gb.ItemSampler(valid_set, batch_size=1000000, shuffle=False)
datapipe = datapipe.sample_neighbor(graph, [4, 4])
datapipe = datapipe.fetch_feature(feature, node_feature_keys=["features"])
datapipe = datapipe.copy_to(device)
valid_dataloader = gb.DataLoader(datapipe, num_workers=0)

In [183]:
EPOCHS = 10

metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)

for epoch in range(EPOCHS):
    metric = metric.to(device)

    model.train()
    
    with tqdm(train_dataloader) as tq:
        
        for step, data in enumerate(tq):
            
            x = data.node_features["features"]
            labels = data.labels
            
            predictions = model(data.blocks, x)
            
            loss = F.cross_entropy(predictions, labels)
            opt.zero_grad()
            loss.backward()
            opt.step()
            
            accuracy = metric(predictions, labels)
            
            tq.set_postfix(
                {"loss": f"{loss.detach()}", "acc": f"{accuracy.detach()}"}
            )
            
    model.eval()
    predictions = []
    labels = []
    
    metric = metric.to("cpu")
    with tqdm(valid_dataloader) as tq, torch.no_grad(): # two context managers
        for data in tq:
            x = data.node_features["features"]
            labels.append(data.labels.detach().cpu())
            predictions.append(model(data.blocks, x).argmax(1).detach().cpu())
        predictions = torch.concatenate(predictions)
        labels = torch.concatenate(labels)
        accuracy = metric(predictions, labels)
        print("Epoch {} Validation Accuracy {}".format(epoch, accuracy))

        # Note that this tutorial do not train the whole model to the end.

2it [00:01,  1.67it/s, loss=0.11816471815109253, acc=0.9596281051635742]
1it [00:00,  3.05it/s]


Epoch 0 Validation Accuracy 0.9549524784088135


2it [00:01,  1.79it/s, loss=0.11270258575677872, acc=0.9606998562812805]
1it [00:00,  3.01it/s]


Epoch 1 Validation Accuracy 0.9543036818504333


2it [00:01,  1.84it/s, loss=0.11627937853336334, acc=0.9599001407623291]
1it [00:00,  2.82it/s]


...

Epoch 6 Validation Accuracy 0.9557971954345703


2it [00:01,  1.91it/s, loss=0.11011911928653717, acc=0.9616246819496155]
1it [00:00,  2.95it/s]


Epoch 7 Validation Accuracy 0.955716073513031


2it [00:01,  1.86it/s, loss=0.1076512336730957, acc=0.962794303894043]  
1it [00:00,  2.80it/s]


KeyboardInterrupt: 

In [189]:
from dgl import remove_self_loop

In [190]:
remove_self_loop(graph)

AttributeError: 'FusedCSCSamplingGraph' object has no attribute 'to_canonical_etype'